# 직접 해보기 00 · 연구 동향을 다시 계산하기

《우주 리서치》 서문의 동향 그래프를, 연도별 논문 기록 수에서 다시 계산한다. 자료는 OpenAlex를 이 책의 검색식으로 집계한 표(2026-10-22 동결본)다.

같은 계산이 `analysis.py`에도 들어 있다. 이 노트북은 그 함수를 불러 단계별로 따라간다.

In [ ]:
import analysis as A
import matplotlib.pyplot as plt

topics = {r["id"]: r for r in A.read("keyword_metrics.csv")}
dens = A.read("denominators.csv")
print("파일 지문", A.sha8("keyword_metrics.csv"), A.sha8("denominators.csv"))
print("열:", list(next(iter(topics.values())).keys()))

## 1. 표를 연다
열 이름을 확인했다. 연도별 건수는 `2021`…`2025`, 검색식은 `data/queries.json`에 따로 있다.

In [ ]:
for k in ("exoplanet_atmosphere", "gravitational_wave", "lrd"):
    print(topics[k]["label_ko"], [int(topics[k][y]) for y in A.YEARS])

## 2. 증가율 — (2024+2025)÷(2021+2022)−1

In [ ]:
t = A.trend(topics["gravitational_wave"], dens, ["2021", "2022"], ["2024", "2025"], "science")
print(t["early"], "→", t["recent"], f"{t['growth_pct']:+.1f}%")

## 3. 기준선 — 전체의 증가율

In [ ]:
print(t["den_early"], "→", t["den_recent"], f"{t['den_growth_pct']:+.1f}%")

## 4. 비중으로 바꾼다
건수는 늘었는데 비중은 그대로인 주제가 있는가?

In [ ]:
for k in ("exoplanet_atmosphere", "gravitational_wave"):
    t = A.trend(topics[k], dens, ["2021", "2022"], ["2024", "2025"], "science")
    print(f"{t['label']}: {t['growth_pct']:+.1f}%  비중 {t['share_early_pct']:.2f}% → {t['share_recent_pct']:.2f}%")

In [ ]:
# 서문 그래프처럼: 2021=1로 맞춘 연도별 건수 (점선은 전체)
den = [int(r["science"]) for r in dens if r["year"] in A.YEARS]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(A.YEARS, [v / den[0] for v in den], "--", color="gray", label="전체(분모)")
for k in ("gravitational_wave", "exoplanet_atmosphere", "lunar_samples"):
    y = [int(topics[k][yr]) for yr in A.YEARS]
    ax.plot(A.YEARS, [v / y[0] for v in y], marker="o", label=topics[k]["label_ko"])
ax.set_ylabel("2021년=1"); ax.legend(); plt.show()

## 5. 스스로 묻는다
`lrd`(작은 붉은 점)는 2021–22년이 0건이다. 증가율은 계산할 수 없고, 연도별 건수를 그대로 보여 주는 것이 정직하다.

In [ ]:
t = A.trend(topics["lrd"], dens, ["2021", "2022"], ["2024", "2025"], "science")
print(t["growth_pct"], t["yearly"])

## 6. 조건을 바꿔 본다
연도 묶음과 분모를 바꾸면 값이 어떻게 달라지는지 적어 두자. 분모를 `all`로 바꾸면 공학이 섞인다.

In [ ]:
for early, recent, den in [(["2021","2022"],["2024","2025"],"science"), (["2022","2023"],["2024","2025"],"science"), (["2021","2022"],["2024","2025"],"all")]:
    t = A.trend(topics["gravitational_wave"], dens, early, recent, den)
    print(early, recent, den, f"{t['growth_pct']:+.1f}%", f"비중 {t['share_early_pct']:.2f}→{t['share_recent_pct']:.2f}%")